<a href="https://colab.research.google.com/github/rm-cg/commercial-energy-optimization-synthetic-data/blob/main/Deliverable_1_Mathematical_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mathematical Framework For The Energy Consumption Optimization Synthetic Data Project**


---


To generate a high-quality synthetic dataset that realistically mirrors real-world commercial energy systems. By mathematically modeling the relationships between human heat generation, weather changes, HVAC capacity, and dynamic utility pricing, this document serves as the exact blueprint for a Python data generator.

# Deliverable 1: Data Dictionary & Table Mapping

## 1. Table: buildings

* **building_id** (Integer): A unique identifier for each commercial building.
  * **Range:** Strictly > 0.
  * **Source:** System generated.

* **primary_activity** (String): The main use of the building (e.g., 'Office', 'Retail').
  * **Range:** Must be from a predefined list.
  * **Source:** ASHRAE 90.1 Prototype building classifications.

* **floor_area_sqft** (Float): The total square footage of the building.
  * **Range:** Strictly > 0 (cannot have negative space).
  * **Source:** Base building assumptions.

* **base_hvac_capacity** (Float): The designed cooling capacity for the building's AC system.
  * **Range:** Strictly > 0.
  * **Source:** Derived from ASHRAE engineering standards.

## 2. Table: meters

* **meter_id** (Integer): A unique ID for the energy meter.
  * **Range:** Strictly > 0.
  * **Source:** System generated.

* **building_id** (Integer): Foreign key connecting the meter to a specific building.
  * **Range:** Strictly > 0.
  * **Source:** System generated.

* **utility_type** (String): The type of energy being measured.
  * **Range:** Strictly limited to 'electricity'.
  * **Source:** Standard commercial utility types.

* **is_broken** (Integer): A flag to indicate if a sensor is failing (Real-world messiness).
  * **Range:** Strictly 0 (functional) or 1 (broken).
  * **Source:** Custom synthetic anomaly rules.

## 3. Table: meter_readings

* **reading_id** (Integer): A unique ID for the specific hourly log.
  * **Range:** Strictly > 0.
  * **Source:** System generated.

* **meter_id** (Integer): Foreign key connecting to the meter.
  * **Range:** Strictly > 0.
  * **Source:** System generated.

* **timestamp** (Datetime): The exact date and hour the energy was consumed.
  * **Range:** Valid 24-hour clock formats only.
  * **Source:** System generated timeline.

* **energy_consumed** (Float): The total energy used during that hour.
  * **Range:** Strictly >= 0 (energy consumed cannot be negative).
  * **Source:** Custom mathematical load equations.

* **ambient_temp_c** (Float): The outside weather temperature influencing the HVAC load.
  * **Range:** 20.0 to 40.0 (Realistic bounds).
  * **Source:** Custom sine-wave weather formula.

## 4. Table: occupancy

* **occupancy_id** (Integer): A unique ID for the headcount log.
  * **Range:** Strictly > 0.
  * **Source:** System generated.

* **building_id** (Integer): Foreign key connecting to the building.
  * **Range:** Strictly > 0.
  * **Source:** System generated.

* **timestamp** (Datetime): The exact date and hour of the headcount.
  * **Range:** Valid datetime matching the meter readings.
  * **Source:** System generated timeline.

* **headcount** (Integer): The number of humans inside the building generating heat.
  * **Range:** Strictly >= 0 (cannot have negative people).
  * **Source:** Custom probability curve (Poisson distribution).

## 5. Table: tariffs

* **tariff_id** (Integer): A unique ID for the pricing rule.
  * **Range:** Strictly > 0.
  * **Source:** System generated.

* **time_of_use_tier** (String): The pricing tier based on the time of day.
  * **Range:** Strictly limited to 'Peak' or 'Off-Peak'.
  * **Source:** Local utility pricing structures.

* **rate_per_kwh** (Float): The financial cost of electricity per kilowatt-hour.
  * **Range:** Strictly > 0.
  * **Source:** Local utility commercial rates.

## 6. Omitted Variables

* **faulty_insulation_multiplier** (Float): A hidden variable that simulates degraded building insulation over time.
  * **Rule:** Multiplies the total energy_consumed by 1.15 to 1.30 for a random 20% of the building portfolio.
  * **Export Rule:** STRICTLY DELETE this column from the dataframe before exporting the final CSV. Future students must use EDA to figure out why these specific buildings are wasting energy.

* **unreported_overtime_multiplier** (Float): A hidden variable simulating employees staying late without logging it in the system.
  * **Rule:** Secretly increases total energy consumption by 20% between 18:00 and 21:00 for a random 20% of the commercial facilities."
  * **Export Rule:** STRICTLY DELETE this column before exporting the final CSV.


#### 7. Real-World Messiness (Anomalies & Noise)

*   **Broken Sensors (Missing Data):**
    *   **Rule:** If the `is_broken` flag in the `meters` table equals 1, randomly replace 5% of that meter's `energy_consumed` hourly readings with `NaN` (blank values) to simulate transmission failures.
*   **Measurement Noise:**
    *   **Rule:** Real-world sensors are imperfect. Apply a randomized Gaussian noise variable ($\pm 2\%$) to all final `energy_consumed` calculations.

#### 8. Formulating Custom Mathematical Equations (Physics + Economics Model)

**A. The Human Sensible Heat Load**
Using the ASHRAE baseline, humans act as 100-Watt heat emitters.
<br></br>
$$Q_{human, t} = \frac{Headcount_t \cdot 100 \text{ W}}{1000}$$


---

**B. Thermal Velocity (The Calculus Component)**
The HVAC system's strain is dictated not just by how hot it is, but by *how fast* the temperature is rising. We calculate the derivative of the ambient temperature with respect to time to find the "Thermal Velocity" ($v_{thermal}$).

<br>
$$v_{thermal} = \frac{dT_{ambient}}{dt} \approx T_{ambient, t} - T_{ambient, t-1}$$
</br>

---

**C. The HVAC Kinetic Overdrive Equation**
When the thermal velocity is positive (temperature is rapidly spiking), the HVAC compressor must work in "overdrive" to overcome the building's thermal inertia. The energy required scales exponentially with the velocity.
<br></br>
$$E_{overdrive, t} = \beta \cdot \max(0, v_{thermal})^2$$

<br>
*(Where $\beta$ is a building-specific kinetic resistance coefficient).*
</br>

---

**D. Total Energy Consumed (With Omitted Variable)**
The final energy load combines the base capacity, human heat, static weather differential, and the dynamic overdrive, all multiplied by our hidden "faulty insulation" anomaly.

<br></br>
$$E_{consumed, t} = \left[ Base_{hvac} + Q_{human, t} + \alpha(T_{ambient, t} - 25^\circ\text{C}) + E_{overdrive, t} \right] \cdot \lambda_{faulty}$$
<br>
*(Where $\alpha$ is the static thermal leakage coefficient, and $\lambda_{faulty}$ is the hidden 1.20 insulation penalty).*
</br>


---

**E. Dynamic Financial Equation (Real-Time Pricing)**
 To calculate the financial cost, we strictly apply Real-Time Pricing (RTP). The marginal price jumps to a higher Peak tier if the energy is consumed during grid stress hours, financially penalizing overconsumption.

 <br>

$$\text{Cost}_t = E_{\text{consumed},t} \cdot \text{Rate}_t$$


</br>

<br>
*(Reference: Mohsenian-Rad & Leon-Garcia, Optimal Residential Load Control with Price Prediction, IEEE).*
</br>

#### 9. Manual Paper Test (Fake Numbers)

*   **Scenario:** An office at 2:00 PM. The temperature spiked from 30°C to 35°C in one hour. 50 people are inside. Faulty insulation penalty (1.20). Base rate is 0.10 PHP, High rate (Peak) is 0.30 PHP.

*   **Assumptions:** $\alpha = 1.5$, $\beta = 0.5$, Base HVAC = 20.0 kWh.



---




**1. Human Heat:**

$$Q_{human} = \frac{50 \cdot 100}{1000} = 5.0 \text{ kWh}$$

**2. Static Weather Load:**

$$E_{static} = 1.5 \cdot (35 - 25) = 15.0 \text{ kWh}$$

**3. Thermal Velocity Overdrive:**

$$E_{overdrive} = 0.5 \cdot \max(0, 35 - 30)^2 = 0.5 \cdot (5)^2 = 12.5 \text{ kWh}$$

**4. Subtotal:**

$$E_{subtotal} = 20.0 + 5.0 + 15.0 + 12.5 = 52.5 \text{ kWh}$$

**5. Hidden Penalty:**

$$E_{consumed} = 52.5 \cdot 1.20 = 63.0 \text{ kWh Total Energy Consumed}$$


**6. Financial Impact (Peak RTP Triggered):**

Since 1:00 PM falls during peak operating hours, the higher tier pricing applies.

$$Cost = 63.0 \text{ kWh} × 0.30 Php = 18.90 Php$$

*   **Conclusion:** The calculus-based consumption model integrates perfectly with the time-of-use economic pricing model.


#### 10. Final Administrative Checks & Budget

*   **Feature Count Verification:** The 5 mapped tables contain exactly 21 total features, successfully meeting the project minimum.
*   **Generative AI Budget:** All text fields and categorical variables in this dataset will be generated using free Python libraries (e.g., NumPy, Pandas, Faker) or predefined arrays. Therefore, the Generative AI API budget is kept safely at 0 PHP.